# AI 201 - Programming Assignment 3
## Implementing the Backpropagation Algorithm

**Instructor:** Prospero C. Naval, Jr., Ph.D.  
**Due Date:** 12:00 noon, March 27, 2026     
**Semester:** 2nd Semester, AY 2025–2026

**Author:** Adriane Jone A. Abunda  
**Student No:** 2025-22450

## Data Preprocessing: Handling Class Imbalance with SMOTE

### Dataset Description

The raw dataset (`data.csv`) contains **3,486 samples × 354 features** with **8 classes**. The class distribution is severely imbalanced:

| Class | Count | % of Total |
|---|---|---|
| 1 | 1,625 | 46.6% |
| 2 | 233 | 6.7% |
| 3 | 30 | 0.9% |
| 4 | 483 | 13.9% |
| 5 | 287 | 8.2% |
| 6 | 310 | 8.9% |
| 7 | 52 | 1.5% |
| 8 | 466 | 13.4% |

Class 3 has only 30 instances while Class 1 has 1,625 — a **54:1 imbalance ratio**. Training a neural network directly on this data would bias the model toward majority classes and produce poor recall for minority classes (3, 7).

### Strategy: Synthetic Minority Over-sampling Technique (SMOTE)

SMOTE generates **synthetic samples** for minority classes by interpolating between existing instances and their k-nearest neighbors in feature space. Unlike random duplication, SMOTE creates novel data points along class decision boundaries, which provides richer gradient information during training.

**Implementation note:** `imblearn` is imported **only** for SMOTE data generation. All neural network computations use NumPy exclusively, as required by the assignment.

In [2]:
# Import Libraries

import numpy as np
from imblearn.over_sampling import SMOTE

In [3]:
# Load the dataset

X = np.loadtxt('data.csv', delimiter=',')
y = np.loadtxt('data_labels.csv', delimiter=',')
print(f"Original Dataset Shape: X={X.shape}, y={y.shape}")

print("\nOriginal Class Distribution")
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {int(c)}: {count} instances")

Original Dataset Shape: X=(3486, 354), y=(3486,)

Original Class Distribution
Class 1: 1625 instances
Class 2: 233 instances
Class 3: 30 instances
Class 4: 483 instances
Class 5: 287 instances
Class 6: 310 instances
Class 7: 52 instances
Class 8: 466 instances


In [4]:
# Apply SMOTE

smote = SMOTE(random_state=42) # random_state for reproducibility
X_balanced, y_balanced = smote.fit_resample(X, y)
print(f"Balanced Dataset Shape: X={X_balanced.shape}, y={y_balanced.shape}")

print("\nBalanced Class Distribution")
classes_bal, counts_bal = np.unique(y_balanced, return_counts=True)
for c, count in zip(classes_bal, counts_bal):
    print(f"Class {int(c)}: {count} instances")


np.random.seed(42) # For reproducibility
indices = np.random.permutation(len(X_balanced))

X_shuffled = X_balanced[indices]
y_shuffled = y_balanced[indices]

def one_hot_encode(labels, num_classes=8):
    """Converts integer labels (1 to num_classes) to one-hot encoded vectors."""
    # Create a zero matrix of shape (number_of_samples, num_classes)
    one_hot = np.zeros((labels.size, num_classes))
    # Subtract 1 from labels to get 0-based indices
    # Advanced indexing: arrange selects rows, labels-1 selects the column to set to 1
    one_hot[np.arange(len(labels)), (labels - 1).astype(int)] = 1.0
    return one_hot

y_shuffled_ohe = one_hot_encode(y_shuffled, num_classes=8)

Balanced Dataset Shape: X=(13000, 354), y=(13000,)

Balanced Class Distribution
Class 1: 1625 instances
Class 2: 1625 instances
Class 3: 1625 instances
Class 4: 1625 instances
Class 5: 1625 instances
Class 6: 1625 instances
Class 7: 1625 instances
Class 8: 1625 instances


### SMOTE Result

After SMOTE, all 8 classes have exactly **1,625 samples**, giving a total of **13,000 samples**. The majority class (Class 1) is unchanged; all other classes are upsampled with synthetic instances.

The dataset is then **randomly shuffled** (`np.random.seed(42)` for reproducibility) and **one-hot encoded** for compatibility with the neural network's 8-output softmax-like output layer.

In [5]:
# Split into train and validation sets
# Valdation set must contain exactly 800 samples
val_size = 800

X_val = X_shuffled[:val_size]
y_val_ohe = y_shuffled_ohe[:val_size]

X_train = X_shuffled[val_size:]
y_train_ohe = y_shuffled_ohe[val_size:]

print(f"Training Set Shape: X={X_train.shape}, y={y_train_ohe.shape}")
print(f"Validation Set Shape: X={X_val.shape}, y={y_val_ohe.shape}")

# Summing the columns of the one-hot encoded validation labels
val_class_counts = np.sum(y_val_ohe, axis=0)
print(f"Validation Class Distribution: {val_class_counts}")

# Save to CSV
np.savetxt('training_set.csv', X_train, delimiter=',', fmt='%.6f')
np.savetxt('training_labels.csv', y_train_ohe, delimiter=',', fmt='%d')

np.savetxt('validation_set.csv', X_val, delimiter=',', fmt='%.6f')
np.savetxt('validation_labels.csv', y_val_ohe, delimiter=',', fmt='%d')

Training Set Shape: X=(12200, 354), y=(12200, 8)
Validation Set Shape: X=(800, 354), y=(800, 8)
Validation Class Distribution: [112. 108.  97.  88. 116.  92.  89.  98.]


### Train/Validation Split

The shuffled dataset is partitioned as:
- **Validation set**: first 800 samples → `validation_set.csv` / `validation_labels.csv`
- **Training set**: remaining 12,200 samples → `training_set.csv` / `training_labels.csv`

Because the data was shuffled from a perfectly balanced distribution (1,625 per class), taking the first 800 samples yields approximately **100 samples per class** in the validation set — close to uniform as required.

All four CSV files are passed to the Neural Network notebook for training and evaluation.